# Women Safety ML Project — Data Analysis & Preprocessing

**Workflow followed:**  
`Raw Dataset → Load Dataset → Basic Data Understanding → Data Analysis + Visualization → Data Cleaning → Missing Values → Outlier Analysis → Categorical Encoding → Feature Scaling → ML Ready`

This notebook follows the uploaded **Machine Learning Project Workflow** documentation. The documentation recommends `SimpleImputer` for missing values, IQR for outlier investigation, `OrdinalEncoder`/`OneHotEncoder` for categorical variables, and `StandardScaler`/`MinMaxScaler` for scaling. 


## 1. Import Libraries and Load Dataset

The dataset is loaded with Pandas as specified in the workflow documentation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)

# Update this path if the CSV is stored in another location.
DATA_PATH = "women_safety_dataset.csv"

df = pd.read_csv(DATA_PATH)
df.head()


## 2. Basic Data Understanding

In [ ]:
print("Shape:", df.shape)
print("\nData types and non-null counts:")
df.info()


In [ ]:
print("Descriptive statistics:")
display(df.describe(include="all").T)


In [ ]:
print("Missing values:")
display(df.isnull().sum().sort_values(ascending=False).to_frame("missing_count"))

print("\nDuplicate rows:", df.duplicated().sum())

print("\nUnique values:")
display(df.nunique().sort_values(ascending=False).to_frame("unique_count"))


In [ ]:
# Categorical frequency tables
categorical_cols = df.select_dtypes(include=["object", "category", "bool"]).columns

for col in categorical_cols:
    print(f"\n--- {col} ---")
    display(df[col].value_counts(dropna=False).to_frame("count"))


## 3. Data Cleaning — Duplicate Rows

In [ ]:
before = len(df)
df = df.drop_duplicates().copy()
after = len(df)

print("Rows before:", before)
print("Rows after :", after)
print("Duplicates removed:", before - after)


## 4. Visualization

In [ ]:
# Missing-value visualization
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

if len(missing):
    plt.figure(figsize=(8, 4))
    missing.plot(kind="bar")
    plt.title("Missing Values by Feature")
    plt.ylabel("Missing count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found.")


In [ ]:
# Risk-level distribution
if "risk_level" in df.columns:
    order = ["Low", "Medium", "High"]
    df["risk_level"].value_counts().reindex(order).plot(kind="bar", figsize=(6, 4))
    plt.title("Risk Level Distribution")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()


In [ ]:
# Histograms for numerical variables
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

for col in numeric_cols:
    plt.figure(figsize=(6, 4))
    df[col].plot(kind="hist", bins=15)
    plt.title(f"Distribution: {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


In [ ]:
# Boxplots for numerical variables
if numeric_cols:
    plt.figure(figsize=(12, 5))
    df[numeric_cols].boxplot(rot=60)
    plt.title("Boxplots of Numerical Features")
    plt.ylabel("Value")
    plt.tight_layout()
    plt.show()


In [ ]:
# Correlation heatmap using matplotlib
if numeric_cols:
    corr = df[numeric_cols].corr()

    plt.figure(figsize=(10, 8))
    plt.imshow(corr, aspect="auto")
    plt.colorbar(label="Correlation")
    plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=70, ha="right")
    plt.yticks(range(len(numeric_cols)), numeric_cols)
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.show()


## 5. Missing Value Handling

Following the documentation:
- Numerical features → **median** imputation
- Categorical features → **most frequent** imputation

The documentation specifically recommends `SimpleImputer` rather than allowing missing values to reach models that cannot handle them.


### Check missing values before preprocessing

In [ ]:
display(df.isnull().sum().sort_values(ascending=False).to_frame("missing_before"))


## 6. Outlier Analysis — IQR Method

For each numerical feature:

- Q1 = 25th percentile
- Q3 = 75th percentile
- IQR = Q3 − Q1
- Lower Bound = Q1 − 1.5 × IQR
- Upper Bound = Q3 + 1.5 × IQR

The workflow documentation says outliers should be investigated before removing them, because an outlier can be a legitimate observation.


In [ ]:
outlier_rows = []

for col in numeric_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    count = int(((df[col] < lower) | (df[col] > upper)).sum())

    outlier_rows.append({
        "feature": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": count
    })

outlier_df = pd.DataFrame(outlier_rows)
display(outlier_df.sort_values("outlier_count", ascending=False))


### Outlier decision

Flagged values are **not automatically deleted**. They should first be checked as data-entry errors or legitimate observations, as required by the workflow documentation. For this dataset, the preprocessing pipeline therefore retains the observations and uses robust median imputation for missing numerical values.


## 7. Prepare Features and Candidate Target

For the safety-risk classification task, `risk_level` is used as the **candidate target**:

- Low → 0
- Medium → 1
- High → 2

`risk_score` is excluded from the input features because it directly determines the risk level in this dataset; retaining it would create target leakage.

Identifiers and raw route/location text are also excluded from this baseline feature matrix.


In [ ]:
target = "risk_level"

drop_for_ml = [
    "id",
    "location",
    "safe_route_path",
    "risk_score",
    target
]

X = df.drop(columns=drop_for_ml)
y = df[target].map({"Low": 0, "Medium": 1, "High": 2})

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numerical features:", numeric_features)
print("\nCategorical features:", categorical_features)
print("\nTarget distribution:")
display(df[target].value_counts().to_frame("count"))


## 8. Build the Preprocessing Pipeline

The preprocessing pipeline applies:

1. **Median imputation + StandardScaler** to numerical features.
2. **Most-frequent imputation + OneHotEncoder** to categorical features.

This keeps preprocessing reproducible and prevents inconsistent transformations later when a model is trained.


In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

X_ready = preprocessor.fit_transform(X)

feature_names = preprocessor.get_feature_names_out()
X_ready_df = pd.DataFrame(X_ready, columns=feature_names, index=X.index)

y_encoded = y.astype(int)

ml_ready = X_ready_df.copy()
ml_ready["risk_level_target"] = y_encoded.values

print("ML-ready shape:", ml_ready.shape)
display(ml_ready.head())


## 9. Verify Preprocessing Results

In [ ]:
print("Missing values in ML-ready data:", int(ml_ready.isnull().sum().sum()))
print("ML-ready rows:", ml_ready.shape[0])
print("ML-ready features including target:", ml_ready.shape[1])

print("\nTarget distribution:")
display(y_encoded.value_counts().sort_index().rename(index={0:"Low", 1:"Medium", 2:"High"}).to_frame("count"))


In [ ]:
# Numerical transformed features should have approximately mean 0 and std 1.
scaled_cols = [c for c in ml_ready.columns if c.startswith("num__")]

if scaled_cols:
    scaling_check = pd.DataFrame({
        "mean": ml_ready[scaled_cols].mean(),
        "std": ml_ready[scaled_cols].std(ddof=0)
    })
    display(scaling_check.head(20))


## 10. Save the Preprocessed Dataset

The resulting CSV is ready to be used in the next stage: train/test split and ML model training.


In [ ]:
OUTPUT_PATH = "ml_ready_dataset.csv"
ml_ready.to_csv(OUTPUT_PATH, index=False)

OUTLIER_PATH = "outlier_analysis.csv"
outlier_df.to_csv(OUTLIER_PATH, index=False)

print(f"Saved: {OUTPUT_PATH}")
print(f"Saved: {OUTLIER_PATH}")


# Preprocessing Complete ✅

**Completed:**  
Load Dataset → Understand Data → Data Analysis → Visualization → Handle Missing Values → Analyze Outliers → Encode Categorical Data → Scale Numerical Features → ML Ready

**Next stage:** Train/test split, model training, evaluation, and model comparison.
